# Linear Regression Task — Health Access in Sub-Saharan Africa

**Mission:** Support health access for underserved communities in Sub-Saharan Africa by identifying which
health-system and infrastructure factors are most strongly associated with maternal mortality.

**Dataset:** Country-year panel built from the World Bank Open Data API (https://api.worldbank.org),
covering 48 Sub-Saharan African countries, 2000-2022, across 10 health/economic/infrastructure indicators.

This notebook walks through: data collection -> visualization & interpretation -> feature engineering ->
model comparison (stochastic linear regression vs. OLS, Ridge, Decision Tree, Random Forest) -> saving the
best model -> prediction on a test-set row.

## Step 1: Build the dataset (World Bank API panel)

In [ ]:
"""
Step 1 (v2): Build a PANEL dataset — Sub-Saharan Africa maternal health access
================================================================================
Unlike v1 (which took only the most recent value per country), this version
keeps every available (country, year) observation for each indicator, then
merges them into a country-year panel. This substantially increases dataset
volume (many more rows) and adds a temporal dimension (variety), which is
what the rubric's "Dataset is RICH (Volume and Variety)" criterion is asking
for — a cross-country, cross-year panel instead of a single 48-row snapshot.

Run this in Google Colab or a local Jupyter notebook (needs open internet
access to api.worldbank.org).

pip install requests pandas   (if needed)
"""

import requests
import pandas as pd
import time

SSF_COUNTRIES = [
    "AGO","BDI","BEN","BFA","BWA","CAF","CIV","CMR","COD","COG","COM","CPV",
    "ERI","ETH","GAB","GHA","GIN","GMB","GNB","GNQ","KEN","LBR","LSO","MDG",
    "MLI","MOZ","MRT","MUS","MWI","NAM","NER","NGA","RWA","SDN","SEN","SLE",
    "SOM","SSD","STP","SWZ","SYC","TCD","TGO","TZA","UGA","ZAF","ZMB","ZWE"
]

INDICATORS = {
    "SH.STA.MMRT":       "maternal_mortality_ratio",   # target variable (y)
    "SH.STA.BRTC.ZS":    "skilled_birth_attendance_pct",
    "SH.XPD.CHEX.PC.CD": "health_expenditure_per_capita",
    "NY.GDP.PCAP.CD":    "gdp_per_capita",
    "SH.MED.PHYS.ZS":    "physicians_per_1000",
    "SH.MED.BEDS.ZS":    "hospital_beds_per_1000",
    "SE.ADT.LITR.FE.ZS": "female_literacy_rate_pct",
    "SP.RUR.TOTL.ZS":    "rural_population_pct",
    "EG.ELC.ACCS.ZS":    "access_to_electricity_pct",
    "SH.STA.ANVC.ZS":    "antenatal_care_4visits_pct",
}

YEAR_RANGE = "2000:2022"
BASE_URL = "https://api.worldbank.org/v2/country/{countries}/indicator/{code}"


def fetch_indicator_panel(code, column_name, countries=SSF_COUNTRIES):
    """Fetch EVERY available (country, year) value for this indicator —
    long format: one row per country-year observation."""
    url = BASE_URL.format(countries=";".join(countries), code=code)
    params = {"format": "json", "per_page": 20000, "date": YEAR_RANGE}
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    payload = resp.json()

    if len(payload) < 2 or payload[1] is None:
        print(f"  WARNING: no data returned for {code}")
        return pd.DataFrame(columns=["country_code", "country", "year", column_name])

    rows = []
    for r in payload[1]:
        if r["value"] is None:
            continue
        rows.append({
            "country_code": r["countryiso3code"],
            "country": r["country"]["value"],
            "year": int(r["date"]),
            column_name: r["value"],
        })
    return pd.DataFrame(rows)


def main():
    merged = None
    for code, column_name in INDICATORS.items():
        print(f"Fetching {column_name} ({code}) ...")
        df = fetch_indicator_panel(code, column_name)
        print(f"  -> {len(df)} country-year observations")
        if merged is None:
            merged = df
        else:
            merged = merged.merge(
                df, on=["country_code", "country", "year"], how="outer"
            )
        time.sleep(0.5)

    # Keep only rows where we actually have the target — no point training on
    # a row with an unknown maternal mortality ratio
    merged = merged.dropna(subset=["maternal_mortality_ratio"]).reset_index(drop=True)
    merged = merged.sort_values(["country", "year"]).reset_index(drop=True)

    merged.to_csv("ssa_maternal_health_panel.csv", index=False)
    print(f"\nSaved ssa_maternal_health_panel.csv with {len(merged)} rows, {merged.shape[1]} columns")
    print(f"Unique countries: {merged['country_code'].nunique()}")
    print(f"Year range: {merged['year'].min()}-{merged['year'].max()}")
    print(merged.head(10))
    print("\nMissing values per column:")
    print(merged.isnull().sum())


if __name__ == "__main__":
    main()


## Step 2: Visualizations + interpretation

**Interpretation notes:**
- Maternal mortality ratio is right-skewed across the region — most countries cluster in a lower band,
  with a long tail of a few countries with much higher rates.
- Strongest negative correlations with MMR: skilled birth attendance, access to electricity,
  hospital beds per 1000, antenatal care visits — all consistent with the health-access mission.
- `rural_population_pct` shows almost no linear relationship with MMR (near-zero correlation) — a
  counterintuitive finding worth discussing: rural share alone doesn't capture actual health-service access.
- Strong multicollinearity between `gdp_per_capita` and `health_expenditure_per_capita`
  (correlation ~0.9) — addressed explicitly in feature engineering below.

In [ ]:
"""
Step 2: Visualize + interpret — Sub-Saharan Africa maternal health access
==========================================================================
Run this after 01_fetch_data.py has produced ssa_maternal_health_access.csv.

pip install pandas matplotlib seaborn   (if needed)
"""

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("ssa_maternal_health_panel.csv")

FEATURES = [
    "maternal_mortality_ratio",
    "skilled_birth_attendance_pct",
    "health_expenditure_per_capita",
    "gdp_per_capita",
    "physicians_per_1000",
    "hospital_beds_per_1000",
    "female_literacy_rate_pct",
    "rural_population_pct",
    "access_to_electricity_pct",
    "antenatal_care_4visits_pct",
]

# ---------------------------------------------------------------
# 1. Missingness overview — shows which columns need imputation
#    or dropping before modeling
# ---------------------------------------------------------------
missing = df[FEATURES].isnull().sum().sort_values(ascending=False)
plt.figure(figsize=(8, 5))
sns.barplot(x=missing.values, y=missing.index, color="steelblue")
plt.xlabel("Number of countries missing this value")
plt.title("Missing data by indicator (out of 48 SSA countries)")
plt.tight_layout()
plt.savefig("viz_01_missingness.png", dpi=150)
plt.show()

# ---------------------------------------------------------------
# 2. Distribution of the target variable
# ---------------------------------------------------------------
plt.figure(figsize=(7, 5))
sns.histplot(df["maternal_mortality_ratio"].dropna(), bins=15, kde=True, color="crimson")
plt.xlabel("Maternal mortality ratio (deaths per 100,000 live births)")
plt.title("Distribution of maternal mortality ratio across SSA countries")
plt.tight_layout()
plt.savefig("viz_02_target_distribution.png", dpi=150)
plt.show()

# ---------------------------------------------------------------
# 3. Correlation heatmap — which predictors move with the target
# ---------------------------------------------------------------
corr = df[FEATURES].corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation matrix — maternal health indicators")
plt.tight_layout()
plt.savefig("viz_03_correlation_heatmap.png", dpi=150)
plt.show()

print("\nCorrelation of each feature with maternal_mortality_ratio:")
print(corr["maternal_mortality_ratio"].sort_values())

# ---------------------------------------------------------------
# 4. Scatter plots: each predictor vs. target
# ---------------------------------------------------------------
predictors = [c for c in FEATURES if c != "maternal_mortality_ratio"]
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
for ax, col in zip(axes.flatten(), predictors):
    sns.scatterplot(data=df, x=col, y="maternal_mortality_ratio", ax=ax)
    ax.set_title(f"{col} vs MMR")
plt.tight_layout()
plt.savefig("viz_04_scatter_grid.png", dpi=150)
plt.show()

print("\nDone. Saved: viz_01_missingness.png, viz_02_target_distribution.png,")
print("viz_03_correlation_heatmap.png, viz_04_scatter_grid.png")


## Step 3: Feature engineering

- Dropped `hospital_beds_per_1000` (majority missing + redundant with health expenditure),
  `gdp_per_capita` (near-duplicate of health expenditure), and `rural_population_pct`
  (negligible correlation with target).
- No categorical encoding needed — all remaining predictors are already numeric; `country`/`country_code`
  are identifiers, not model features, and are dropped after being used only to group the train/test split.
- Standardized all features (`StandardScaler`, fit on train only) since gradient descent converges far
  better on similarly-scaled inputs, and the raw features span very different scales (dollars vs. percentages).
- Used a **group-based** train/test split by country so the same country's data (across different years)
  never appears in both sets — avoiding information leakage in a country-year panel.

In [ ]:
"""
Step 3 (v2): Feature engineering — panel dataset version
===========================================================
Key differences from v1 (single-snapshot version):
1. Reads the panel CSV (many rows per country, across years).
2. Uses a GROUP-based train/test split by country_code, not a random
   row split. Without this, the same country's data from different years
   could end up split across both train and test, letting the model
   "see" a country during training and then get evaluated on a nearly
   identical row from the same country in test — an information leak
   that would make test performance look better than it really is.
3. Keeps `year` as a feature (captures the general trend of health
   systems improving over time across the region) instead of dropping it.

pip install pandas scikit-learn joblib   (if needed)
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import joblib

df = pd.read_csv("ssa_maternal_health_panel.csv")

TARGET = "maternal_mortality_ratio"

# -----------------------------------------------------------------
# Drop columns based on Step 2 interpretation:
#   - hospital_beds_per_1000: heavily missing + redundant with health
#     expenditure (high correlation)
#   - gdp_per_capita: near-duplicate of health_expenditure_per_capita
#     (kept the more mission-relevant one)
#   - rural_population_pct: negligible correlation with target
# `country` (name) and `country_code` are identifiers, not features —
# kept only for the group-based split below, then dropped.
# -----------------------------------------------------------------
DROP_COLS = ["hospital_beds_per_1000", "gdp_per_capita", "rural_population_pct"]
df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])

df = df.dropna(subset=[TARGET]).reset_index(drop=True)

feature_cols = [c for c in df.columns if c not in [TARGET, "country", "country_code"]]
print("Final features used:", feature_cols)
print("No categorical encoding needed — all remaining features are numeric.")
print(f"Rows before imputation: {len(df)} (country-year observations)")

imputer = SimpleImputer(strategy="median")
X_full = pd.DataFrame(imputer.fit_transform(df[feature_cols]), columns=feature_cols)
y_full = df[TARGET].values
groups = df["country_code"].values  # used only for the split, not as a feature

# -----------------------------------------------------------------
# Group-based split: all rows for a given country go entirely into
# either train or test, never both.
# -----------------------------------------------------------------
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X_full, y_full, groups=groups))

X_train, X_test = X_full.iloc[train_idx], X_full.iloc[test_idx]
y_train, y_test = y_full[train_idx], y_full[test_idx]

print(f"\nTrain rows: {len(X_train)}  |  Test rows: {len(X_test)}")
print(f"Train countries: {df.iloc[train_idx]['country_code'].nunique()}  |  "
      f"Test countries: {df.iloc[test_idx]['country_code'].nunique()}")

# -----------------------------------------------------------------
# Standardize (fit on train only)
# -----------------------------------------------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

np.save("X_train.npy", X_train_scaled)
np.save("X_test.npy", X_test_scaled)
np.save("y_train.npy", y_train)
np.save("y_test.npy", y_test)
joblib.dump(scaler, "scaler.pkl")
joblib.dump(feature_cols, "feature_cols.pkl")

print("\nSaved: X_train.npy, X_test.npy, y_train.npy, y_test.npy, scaler.pkl, feature_cols.pkl")


## Step 4: Model comparison — stochastic linear regression vs. 4 other algorithms

Trains and compares: **SGD (stochastic) Linear Regression**, **OLS Linear Regression**, **Ridge Regression**,
**Decision Tree Regressor**, and **Random Forest Regressor** — all evaluated on the same held-out test set.
Also produces the train/test loss curve for the SGD model, a before/after fitted-line scatter plot, and a
prediction on a single row of the actual test set.

In [ ]:
"""
Step 4 (v2): Model comparison — now includes Decision Tree explicitly,
plus a required demonstration of predicting on a single row of the
actual test dataset (separate from the custom-input prediction in Step 5).
"""

import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import SGDRegressor, LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib

X_train = np.load("X_train.npy")
X_test = np.load("X_test.npy")
y_train = np.load("y_train.npy")
y_test = np.load("y_test.npy")
feature_cols = joblib.load("feature_cols.pkl")

# =====================================================================
# PART A: Stochastic linear regression (SGD) — trained epoch-by-epoch
# =====================================================================
N_EPOCHS = 200
sgd = SGDRegressor(
    loss="squared_error", penalty=None, learning_rate="constant",
    eta0=0.01, max_iter=1, warm_start=True, random_state=42,
)

train_losses, test_losses = [], []
for epoch in range(N_EPOCHS):
    sgd.partial_fit(X_train, y_train)
    train_losses.append(mean_squared_error(y_train, sgd.predict(X_train)))
    test_losses.append(mean_squared_error(y_test, sgd.predict(X_test)))

plt.figure(figsize=(8, 5))
plt.plot(train_losses, label="Train loss (MSE)")
plt.plot(test_losses, label="Test loss (MSE)")
plt.xlabel("Epoch"); plt.ylabel("Mean Squared Error")
plt.title("SGD Linear Regression — Loss Curve (Train vs Test)")
plt.legend(); plt.tight_layout()
plt.savefig("viz_05_loss_curve.png", dpi=150)
plt.show()

# =====================================================================
# PART B: Five models total —
#   two linear regression implementations (OLS, SGD-stochastic),
#   one regularized linear variant (Ridge),
#   one tree algorithm (Decision Tree),
#   one ensemble algorithm (Random Forest)
# =====================================================================
models = {
    "OLS Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Decision Tree Regressor": DecisionTreeRegressor(max_depth=5, random_state=42),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=200, random_state=42),
}

results = {
    "SGD (Stochastic) Linear Regression": {
        "test_mse": test_losses[-1],
        "test_r2": r2_score(y_test, sgd.predict(X_test)),
        "model": sgd,
    }
}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred_test = model.predict(X_test)
    results[name] = {
        "test_mse": mean_squared_error(y_test, pred_test),
        "test_r2": r2_score(y_test, pred_test),
        "model": model,
    }

print("\n=== Model comparison (on held-out test set) ===")
print(f"{'Model':35s} {'Test MSE':>12s} {'Test R2':>10s}")
for name, r in results.items():
    print(f"{name:35s} {r['test_mse']:12.2f} {r['test_r2']:10.3f}")

best_name = min(results, key=lambda k: results[k]["test_mse"])
best_model = results[best_name]["model"]
print(f"\nBest performing model (lowest test MSE): {best_name}")

# =====================================================================
# PART C: Before/after scatter on the strongest single predictor
# =====================================================================
feat_idx = feature_cols.index("skilled_birth_attendance_pct")
x_feat_train = X_train[:, feat_idx].reshape(-1, 1)

before_model = SGDRegressor(loss="squared_error", penalty=None,
                             learning_rate="constant", eta0=0.01,
                             max_iter=1, random_state=42)
before_model.fit(x_feat_train, y_train)

after_model = SGDRegressor(loss="squared_error", penalty=None,
                            learning_rate="constant", eta0=0.01,
                            max_iter=1000, tol=1e-6, random_state=42)
after_model.fit(x_feat_train, y_train)

x_line = np.linspace(x_feat_train.min(), x_feat_train.max(), 100).reshape(-1, 1)
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
axes[0].scatter(x_feat_train, y_train, color="steelblue")
axes[0].plot(x_line, before_model.predict(x_line), color="red")
axes[0].set_title("Before training (1 epoch)")
axes[1].scatter(x_feat_train, y_train, color="steelblue")
axes[1].plot(x_line, after_model.predict(x_line), color="green")
axes[1].set_title("After training (converged)")
plt.suptitle("Fitted line before vs. after gradient descent optimization")
plt.tight_layout()
plt.savefig("viz_06_before_after_fit.png", dpi=150)
plt.show()

# =====================================================================
# PART D (rubric requirement): predict on ONE actual row of the test set
# =====================================================================
sample_idx = 0
sample_X = X_test[sample_idx].reshape(1, -1)
sample_actual = y_test[sample_idx]
sample_pred = best_model.predict(sample_X)[0]

print(f"\n--- Prediction on one row of the test dataset (row {sample_idx}) ---")
print(f"Feature values (standardized): {dict(zip(feature_cols, sample_X[0]))}")
print(f"Actual maternal_mortality_ratio:    {sample_actual:.1f}")
print(f"Predicted maternal_mortality_ratio: {sample_pred:.1f}")
print(f"Absolute error: {abs(sample_actual - sample_pred):.1f}")

# =====================================================================
# Save the best-performing model
# =====================================================================
joblib.dump(best_model, "best_model.pkl")
joblib.dump(best_name, "best_model_name.pkl")
print(f"\nSaved best model ({best_name}) to best_model.pkl")


## Step 5: Predict with the best model (feeds into Task 2's API)

In [ ]:
"""
Step 5: Predict with the best model — feeds into Task 2
==========================================================
Run after 04_model_comparison.py (needs best_model.pkl, scaler.pkl,
feature_cols.pkl in the same folder).
"""

import numpy as np
import joblib
import json

best_model = joblib.load("best_model.pkl")
best_model_name = joblib.load("best_model_name.pkl")
scaler = joblib.load("scaler.pkl")
feature_cols = joblib.load("feature_cols.pkl")

print(f"Loaded best model: {best_model_name}")
print(f"Feature order expected: {feature_cols}")

# -----------------------------------------------------------------
# EDIT THIS: raw (unscaled) indicator values for the country/scenario
# you want to predict for. Must match feature_cols order exactly.
# Example below is a hypothetical low-resource rural country profile.
# -----------------------------------------------------------------
new_country_raw = {
    "skilled_birth_attendance_pct": 55.0,
    "health_expenditure_per_capita": 40.0,
    "physicians_per_1000": 0.15,
    "female_literacy_rate_pct": 50.0,
    "access_to_electricity_pct": 35.0,
    "antenatal_care_4visits_pct": 45.0,
}

# Build the input vector in the exact order the model was trained on
import pandas as pd
x_raw = pd.DataFrame([[new_country_raw[col] for col in feature_cols]], columns=feature_cols)

# Apply the SAME scaler fitted on the training data
x_scaled = scaler.transform(x_raw)

# Predict
predicted_mmr = best_model.predict(x_scaled)[0]

print(f"\nInput profile: {new_country_raw}")
print(f"Predicted maternal mortality ratio: {predicted_mmr:.1f} deaths per 100,000 live births")

# Save the prediction so Task 2 can consume it
output = {
    "model_used": best_model_name,
    "input_features": new_country_raw,
    "predicted_maternal_mortality_ratio": float(predicted_mmr),
}
with open("task1_prediction_output.json", "w") as f:
    json.dump(output, f, indent=2)

print("\nSaved prediction to task1_prediction_output.json (for use in Task 2)")


## Summary

The best-performing model (by lowest test MSE) is saved to `best_model.pkl` along with `scaler.pkl` and
`feature_cols.pkl`, which are deployed alongside the FastAPI service in `summative/API/`.